In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
import torch
from torch import optim
from time import time
import random
import sys
from pathlib import Path
from scipy.interpolate import interp1d

current_dir = Path().resolve()
ModelBase_path = current_dir.parent / 'Model_Base'
sys.path.append(str(ModelBase_path))
from Model import ResNetGELU
print(ModelBase_path)

In [ ]:
def correct_velocity_rozos(
    Ux,
    n: int = 5,
    uncertainty: float = 0.20,
    corr_length: float = 0.30,
    ia_skew: float = 0.10,
    bias_scale: float = 0.03,
    xcorr_low_vel_frac: float = 0.10,
    ci: float = 0.90,
    seed: int | None = None,
) -> dict:
    """
    Rozos et al. (2020) Monte Carlo mean correction.

    Principle
    --------
    For each of the three parameters (contrast adjustment, minimum cross‑correlation,
    IA size), generate n samples from a triangular distribution. Take the n³ Cartesian
    product, apply the corresponding perturbation to each combination, and then use
    the mean profile over the requested CI (e.g., 90 %) as the corrected best estimate.

    Parameters
    ----------
    Ux          : Original (or single‑noise) velocity profile, length N.
    n           : Number of samples per parameter; total runs = n³.
                  Rozos suggests n=5 (125 runs) is sufficient for convergence.
    uncertainty : Total uncertainty (half‑width of the 90 % CI), default 0.20.
    corr_length : Spatial correlation length for IA size perturbations (relative to
                  cross‑section length).
    ia_skew     : Skewness of IA (positive → bias towards large IA → underestimation).
    bias_scale  : Global bias magnitude caused by contrast adjustment.
    xcorr_low_vel_frac : Additional error amplitude for spurious vectors in low‑speed zones.
    ci          : Confidence interval width, default 0.90 (90 % CI).
    seed        : Random seed.

    Returns
    -------
    dict with keys:
        'mean'        : Mean profile (corrected best estimate, Rozos mean profile)
        'lower'       : Lower bound of the CI
        'upper'       : Upper bound of the CI
        'ci_width'    : Point‑wise confidence interval width (can be used to infer
                        optimal L, see Section 3 of the paper)
        'all_profiles': All MC profiles, shape (n³, N)
    """
    Ux = np.asarray(Ux, dtype=float)
    N = len(Ux)
    rng = np.random.default_rng(seed)

    alpha_lo = (1.0 - ci) / 2.0   # 0.05
    alpha_hi = 1.0 - alpha_lo      # 0.95

    # ── Generate n triangular samples for each of the three parameters ──
    # Parameter 1: contrast adjustment  Tri(0.3, 0.9, 0.7) → global bias factor
    p_contrast = rng.triangular(0.3, 0.7, 0.9, size=n)       # shape (n,)
    bias_vals  = ((p_contrast - 0.6) / 0.3) * bias_scale     # ∈ [-bias_scale, +bias_scale]

    # Parameter 2: minimum cross‑correlation  Tri(0.3, 0.9, 0.6) → low‑speed spurious vector intensity
    p_xcorr    = rng.triangular(0.3, 0.6, 0.9, size=n)
    xcorr_vals = ((p_xcorr - 0.6) / 0.3) * xcorr_low_vel_frac

    # Parameter 3: IA size  Tri(0.5L, 2L, L) → spatial correlation length scale
    p_ia       = rng.triangular(0.5, 1.0, 2.0, size=n)       # relative to L
    corr_vals  = p_ia * corr_length                           # actual correlation length

    # ── n³ Cartesian product, apply perturbations per group ──
    profiles = np.empty((n ** 3, N), dtype=float)
    vel_threshold = np.percentile(np.abs(Ux), 25)
    low_mask = np.abs(Ux) < vel_threshold

    idx = 0
    for b in bias_vals:
        for xc in xcorr_vals:
            for cl in corr_vals:
                # --- IA perturbation: spatially correlated multiplicative noise ---
                raw_ia = rng.standard_normal(N)
                sigma_px = max(cl * N, 0.5)
                ia_noise = gaussian_filter1d(raw_ia, sigma=sigma_px)
                ia_noise = ia_noise / (ia_noise.std() + 1e-12) * (uncertainty / 3.0)
                # apply ia_skew (bias towards underestimation)
                ia_noise -= ia_skew * (uncertainty / 3.0)
                noise_ia = 1.0 + ia_noise

                # --- Contrast bias: uniform bias over the whole section ---
                noise_contrast = 1.0 + b

                # --- Low cross‑correlation spurious vectors: only in low‑speed zones ---
                noise_xcorr = np.ones(N)
                if low_mask.any():
                    noise_xcorr[low_mask] = 1.0 + (
                        rng.uniform(-1, 1, int(low_mask.sum())) * abs(xc)
                    )

                profiles[idx] = Ux * noise_ia * noise_contrast * noise_xcorr
                idx += 1

    # ── Take the CI mean profile (Rozos: mean of confidence interval) ──
    lower  = np.quantile(profiles, alpha_lo, axis=0)
    upper  = np.quantile(profiles, alpha_hi, axis=0)
    mean   = profiles.mean(axis=0)

    return {
        'mean'        : mean,
        'lower'       : lower,
        'upper'       : upper,
        'ci_width'    : upper - lower,
        'all_profiles': profiles,
    }

def smooth_velocity(Ux_noisy, method='savgol', window=7, polyorder=3, sigma=2):
    if method == 'savgol':
        win = min(window, len(Ux_noisy) if len(Ux_noisy) %2 != 0
                  else len(Ux_noisy) - 1)
        return savgol_filter(Ux_noisy, window_length=win, polyorder=polyorder)
    elif method == 'gaussian':
        return gaussian_filter1d(Ux_noisy, sigma=sigma)
    elif method == 'moving':
        kernel = np.ones(window) / window
        return np.convolve(
            np.pad(Ux_noisy, window // 2, mode='reflect'),
            kernel, mode='valid'
        )[:len(Ux_noisy)]
    else:
        raise ValueError("method必须为 'savgol' | 'gaussian' | 'moving'")

In [ ]:
# ============================================================
# DataLoader
# ============================================================
def load_array(data_arrays, batch_size, is_train=True):
    dataset = torch.utils.data.TensorDataset(*data_arrays)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=is_train)

def fun_calDiff(data, y, num_points):
    if len(data) != len(y):
        raise ValueError("data and y must have the same length")
    indices = np.linspace(0, len(data) - 1, num_points).astype(int)
    data = [data[i] for i in indices]
    y    = [y[i]    for i in indices]
    U_avi, U_diff, Y_diff = [], [], []
    for i in range(len(data) - 1):
        if y[i + 1] == y[i]:
            raise ValueError("y[i+1] - y[i] is zero")
        U_avi.append((data[i + 1] + data[i]) / 2)
        U_diff.append((data[i + 1] - data[i]) / (y[i + 1] - y[i]))
        Y_diff.append(y[i + 1] - y[i])
    return U_avi, U_diff, Y_diff

def fun_calDiff_gai(data, y, num_points=50):
    if len(data) != len(y):
        raise ValueError("data and y must have the same length")
    
    indices = np.linspace(0, len(data) - 1, num_points)

    interp_data = interp1d(np.arange(len(data)), data, kind='linear', fill_value="extrapolate")
    interp_y = interp1d(np.arange(len(y)), y, kind='linear', fill_value="extrapolate")
    data = interp_data(indices)
    y = interp_y(indices)

    U_avi, U_diff, Y_diff = [], [], []

    for i in range(len(data) - 1):
        if y[i + 1] == y[i]:
            raise ValueError("y[i+1] - y[i] is zero, cannot divide by zero")
            # return 100
        U_avi.append((data[i + 1] + data[i]) / 2)
        U_diff.append(abs((data[i + 1] - data[i])) / abs(y[i + 1] - y[i]))
        Y_diff.append(abs(y[i + 1] - y[i]))

    return U_avi, U_diff, Y_diff
# ============================================================
# splitData0
# Noise affects only the training features
# ============================================================
def splitData0(data0, num_points, apply_augmentation=False):
    U0= data0[0]
    H0    = data0[1]
    B0    = data0[int(len(data0) -1)]
    Fr= (U0 / np.sqrt(9.81 * H0))
    X= B0 + 2 * H0
    R     = B0 * H0 / (B0 + 2 * H0)
    Re    = U0 * R /1e-6
    dsize = int((len(data0) - 2) / 2.0)

    # Keep the original Ux0 for computing filtering conditions
    Ux0_raw = data0[2:dsize + 2]

    # Compute filtering statistics from raw data (unaffected by noise)
    aver_raw= sum(Ux0_raw) / len(Ux0_raw)
    Uamx_Uaver_raw = max(Ux0_raw) / aver_raw

    # Add noise only for generating training features, not for filtering conditions
    if apply_augmentation:
        # Ux_noisy_new = add_velocity_uncertainty_rozos(Ux0_raw, uncertainty=0.20, seed=0)
        result = correct_velocity_rozos(Ux0_raw, n=5, uncertainty=0.20, seed=42)
        Ux_noisy_new = result['mean']
        Ux0      = smooth_velocity(Ux_noisy_new, method='savgol', window=7, polyorder=3)
        # Ux0_noisy = add_velocity_uncertainty(Ux0_raw, uncertainty=0.2)
        # Ux0       = smooth_velocity(Ux0_noisy, method='savgol', window=7, polyorder=3)
    else:
        Ux0 = Ux0_raw

    Y = data0[dsize + 2:2 * dsize + 2]

    half_dsize = int(dsize / 2.0)
    inValueU= Ux0[:half_dsize]
    inValueY   = Y[:half_dsize]

    # Use raw values for aver and Uamx_Uaver to keep filtering conditions consistent
    aver       = aver_raw
    Uamx_Uaver = Uamx_Uaver_raw

    U_avi, U_diff, Y_diff = fun_calDiff(inValueU, inValueY, num_points)

    pin_np1= np.array(U_avi).reshape(1, -1)
    pin_np2 = np.array(U_diff).reshape(1, -1)
    pin_np3 = np.array(Y_diff).reshape(1, -1)
    pin_np  = np.vstack((pin_np1, pin_np2, pin_np3))

    target_np = np.array([U0, H0])
    Z         = np.array([U0, H0, B0, Fr, Re, X, R, num_points,aver, Uamx_Uaver])
    return pin_np, target_np, Z

# ============================================================
# genFeaturesLabels
# ============================================================
def genFeaturesLabels(data_set, num_points=5, maxh=10, maxFr=0.99,
                      maxHoverB=1.2, apply_augmentation=False):
    features, labels, z = [], [], []
    cnt = [0, 0, 0, 0]  # filtering statistics

    for data0 in data_set:
        pin_np, target_np, zi = splitData0(
            data0, num_points,
            apply_augmentation=apply_augmentation
        )
        if zi[9] > 1.1:
            cnt[0] += 1; continue
        if zi[1] > maxh:
            cnt[1] += 1; continue
        if zi[3] > maxFr:
            cnt[2] += 1; continue
        if np.log(zi[1] / zi[2]) > maxHoverB:
            cnt[3] += 1; continue
        features.append(pin_np)
        labels.append(target_np)
        z.append(zi)

    print(f"  Filtering stats → Uamx_Uaver>1.1: {cnt[0]} | "
          f"H>{maxh}: {cnt[1]} | "
          f"Fr>{maxFr}: {cnt[2]} | "
          f"log(H/B)>{maxHoverB}: {cnt[3]} | "
          f"Passed: {len(features)}")

    if len(features) == 0:
        raise ValueError(
            f"Filtered dataset is empty!\n"
            f"Filter conditions: maxh={maxh}, maxFr={maxFr}, maxHoverB={maxHoverB}"
        )

    features = torch.from_numpy(np.array(features))
    labels   = torch.from_numpy(np.array(labels))
    z        = torch.from_numpy(np.array(z))
    return features, labels, z


### gai  PredSplitData  genFeaturesAndB

def SplitData0_new(data0, num_points=50, apply_augmentation=False):

    U0= data0[0]
    H0    = data0[1]
    B0    = data0[int(len(data0) -1)]
    Fr= (U0 / np.sqrt(9.81 * H0))
    X= B0 + 2 * H0
    R     = B0 * H0 / (B0 + 2 * H0)
    Re    = U0 * R /1e-6
    dsize = int((len(data0) - 2) / 2.0)

    # Keep the original Ux0 for computing filtering conditions
    Ux0_raw = data0[2:dsize + 2]

    # Compute filtering statistics from raw data (unaffected by noise)
    aver_raw= sum(Ux0_raw) / len(Ux0_raw)
    Uamx_Uaver_raw = max(Ux0_raw) / aver_raw

    # Use raw values for aver and Uamx_Uaver to keep filtering conditions consistent
    aver       = aver_raw
    Uamx_Uaver = Uamx_Uaver_raw
    
    # Add noise only for generating training features, not for filtering conditions
    if apply_augmentation:
        # Ux_noisy_new = add_velocity_uncertainty_rozos(Ux0_raw, uncertainty=0.20, seed=0)
        result = correct_velocity_rozos(Ux0_raw, n=5, uncertainty=0.20, seed=42)
        Ux_noisy_new = result['mean']
        Ux0      = smooth_velocity(Ux_noisy_new, method='savgol', window=7, polyorder=3)
        # Ux0_noisy = add_velocity_uncertainty(Ux0_raw, uncertainty=0.2)
        # Ux0       = smooth_velocity(Ux0_noisy, method='savgol', window=7, polyorder=3)
    else:
        Ux0 = Ux0_raw
    # inValueU = Ux0

    Y = data0[dsize + 2:2 * dsize + 2]

## Data Grade Enhance ————
    max_index = np.argmax(Ux0)
    Bleft = B0* (max_index + 1)/len(Ux0)
    Bright = B0-Bleft
    Uint = Ux0[:max_index + 1]
    Yint = Y[:max_index + 1]
    U_avi,U_diff,Y_diff = fun_calDiff_gai(Uint,Yint,num_points=num_points)


    pin_np1 = np.array(U_avi).reshape(1,-1)
    pin_np2 = np.array(U_diff).reshape(1,-1)
    pin_np3 = np.array(Y_diff).reshape(1,-1)
    pin_npleft = np.vstack((pin_np1, pin_np2, pin_np3))

    Uint = Ux0[max_index:]
    Uint = Uint[::-1]
    Yint = Y[max_index:]
    Yint = Yint[::-1]

    U_avi,U_diff,Y_diff = fun_calDiff_gai(Uint,Yint,num_points=num_points)
    pin_np1 = np.array(U_avi).reshape(1,-1)
    pin_np2 = np.array(U_diff).reshape(1,-1)
    pin_np3 = np.array(Y_diff).reshape(1,-1)
    pin_npright = np.vstack((pin_np1, pin_np2, pin_np3))

## --------------------------------------------------

    target_np = np.array([U0, H0])
    Z         = np.array([U0, H0, B0, Fr, Re, X, R, num_points, aver, Uamx_Uaver])

    return pin_npleft, pin_npright,Bleft,Bright,target_np,Z


def genFeaturesLabels_gai(data_set, num_points=50, maxh=10, maxFr=0.99,
                      maxHoverB=1.2, apply_augmentation=False):
    features, labels, z = [], [], []
    cnt = [0, 0, 0, 0]  # filtering statistics

    for data0 in data_set:
        pin_npleft, pin_npright,Bleft,Bright,target_np,zi = SplitData0_new(
            data0, num_points,
            apply_augmentation=apply_augmentation
        )

        if zi[9] > 1.1:
            cnt[0] += 1; continue
        if zi[1] > maxh:
            cnt[1] += 1; continue
        if zi[3] > maxFr:
            cnt[2] += 1; continue
        if np.log(zi[1] / zi[2]) > maxHoverB:
            cnt[3] += 1; continue
        
        features.append(pin_npleft)
        labels.append(target_np)
        z.append(zi)

        features.append(pin_npright)
        labels.append(target_np)
        z.append(zi)

    print(f"  Filtering stats → Uamx_Uaver>1.1: {cnt[0]} | "
          f"H>{maxh}: {cnt[1]} | "
          f"Fr>{maxFr}: {cnt[2]} | "
          f"log(H/B)>{maxHoverB}: {cnt[3]} | "
          f"Passed: {len(features)}")

    if len(features) == 0:
        raise ValueError(
            f"Filtered dataset is empty!\n"
            f"Filter conditions: maxh={maxh}, maxFr={maxFr}, maxHoverB={maxHoverB}"
        )

    features = torch.from_numpy(np.array(features))
    labels   = torch.from_numpy(np.array(labels))
    z        = torch.from_numpy(np.array(z))
    return features, labels, z

# ============================================================
# test_loss
# ============================================================
def test_loss(NAME, test_iter, cuda=False):
    if cuda:
        net = torch.load(NAME, weights_only=False)
    else:
        net = torch.load(NAME, map_location='cpu', weights_only=False)
    net.eval()
    if cuda:
        net.cuda()
    else:
        net.cpu()

    test_loss_list = []
    for X, y, z in test_iter:
        pin = torch.from_numpy(np.float32(X))
        pin.requires_grad = False
        if cuda:
            pred = net(pin.cuda())
        else:
            pred = net(pin.cpu())
        UH_preds = pred.detach().cpu().numpy()
        for UH_pred,tnp in zip(UH_preds, y):
            U0_pred = UH_pred[0]
            H0_pred = UH_pred[1]
            loss0 = (((U0_pred - tnp[0]) /tnp[0]) ** 2 +
                     ((H0_pred - tnp[1]) / tnp[1]) ** 2) ** 0.5
            test_loss_list.append(loss0)
    return np.mean(test_loss_list)

# ============================================================
# train: train_iter / test_iter are passed from outside, not rebuilt internally
# ============================================================
def train(NAME, net, reload=False, LR=0.0001, n_epochs=200,
          batch_size=30, cuda=False, num_points=50,
          maxh=10, maxFr=0.99, maxHoverB=1.2,
          train_iter=None, test_iter=None):

    if reload:
        if cuda:
            net = torch.load(NAME, weights_only=False)
        else:
            net = torch.load(NAME, map_location='cpu', weights_only=False)
        net.eval()

    if cuda:
        net.cuda()
    else:
        net.cpu()

    loss_fn   = torch.nn.MSELoss(reduction='mean')
    optimizer = optim.Adam(net.parameters(), lr=LR)
    epoch_loss = []

    if not reload:
        lossLog = open(NAME + "_loss.log", "w")
    else:
        lossLog = open(NAME + "_loss.log", "a")

    for epoch in range(n_epochs):
        for param in net.parameters():
            param.requires_grad = True
        optimizer.zero_grad()

        t0 = time()
        train_loss = []

        for X, y, z in train_iter:
            pin = torch.from_numpy(np.float32(X))
            pin.requires_grad = True

            if cuda:
                pred= net(pin.cuda())
                target = y.cuda()
            else:
                pred   = net(pin.cpu())
                target = y.cpu()

            loss = loss_fn(pred, target.float())
            loss.backward()
            train_loss.append(loss.cpu().detach().numpy())

            for param in net.parameters():
                param.requires_grad = True
            optimizer.step()
            optimizer.zero_grad()

        if len(epoch_loss) > 1:
            if epoch_loss[-1] > epoch_loss[-2]:
                optimizer = optim.Adam(net.parameters(), lr=LR)

        epoch_loss.append(np.mean(train_loss))

        torch.save(net, NAME)
        t_loss = test_loss(NAME, test_iter, cuda)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:4d} | "
                  f"Train Loss: {epoch_loss[-1]:.5e} | "
                  f"Test Loss: {t_loss:.5e} | "
                  f"LR: {LR} | "
                  f"Time: {time() - t0:.1f}s")
        lossLog.write("%d %4.5e %4.5e\n" % (epoch, epoch_loss[-1], t_loss))
        lossLog.close()
        lossLog = open(NAME + "_loss.log", "a")

    lossLog.close()
    torch.save(net, NAME)

# ============================================================
# trainpro: data preprocessing is done only once, DataLoader built only once
# ============================================================
def trainpro(trainName, net, num_points=50, maxh=10, maxFr=0.99, maxHoverB=1.2):
    t0 = time()

    global data_set
    data_size= len(data_set)# 5000
    train_size = int(data_size * 0.8)  # 4000

    random.seed(1)
    train_cases = random.sample(list(np.arange(0, data_size)), train_size)
    train_cases.sort()
    test_cases = [i for i in range(data_size) if i not in train_cases]

    train_data = data_set[train_cases]# (4000, 2003)
    test_data  = data_set[test_cases]   # (1000, 2003)

    print(f"data_set shape: {data_set.shape}")
    print(f"train_data shape : {train_data.shape}")
    print(f"test_data shape  : {test_data.shape}\n")

    # Training set: add noise (done only once, fixed in Tensors)
    print(" Preprocessing training set (with noise augmentation, only once)...")
    train_features, train_labels, train_z = genFeaturesLabels_gai(
        train_data, num_points, maxh, maxFr, maxHoverB,
        apply_augmentation=True
    )

    # Validation set: original clean data
    print(" Preprocessing validation set...")
    test_features, test_labels, test_z = genFeaturesLabels_gai(
        test_data, num_points, maxh, maxFr, maxHoverB,
        apply_augmentation=True
    )

    print(f"\n Training set size : {len(train_features)}")
    print(f" Validation set size : {len(test_features)}")

    batch_size = 20
    # DataLoader built only once, reused throughout training
    train_iter = load_array(
        (train_features, train_labels, train_z),
        batch_size, is_train=True
    )
    test_iter = load_array(
        (test_features, test_labels, test_z),
        batch_size, is_train=False
    )

    # Staged learning rates
    stages = [
        (False, 1.0e-4, 200),
        (True,  1.0e-4, 200),
        (True,  1.0e-5, 200),
        (True,  1.0e-6, 200),
        (True,  1.0e-7, 200),]

    for reload, LR, n_epochs in stages:
        print(f"\n{'='*55}")
        print(f"reload={reload} | LR={LR} | epochs={n_epochs}")
        print(f"{'='*55}")
        train(trainName, net,
              reload=reload, LR=LR, n_epochs=n_epochs,
              batch_size=batch_size, cuda=True,
              num_points=num_points, maxh=maxh,
              maxFr=maxFr, maxHoverB=maxHoverB,
              train_iter=train_iter,
              test_iter=test_iter)
        t1 = time()
    print(f"\n Total training time: {t1 - t0:.1f}s")

In [ ]:
# ============================================================
# main train
# ============================================================
Data_path = current_dir.parent.parent / 'Data/datasets/'
file_name= 'Random5000.dat'
file_path  = os.path.join(Data_path, file_name)
data_set   = np.float32(np.loadtxt(file_path, unpack=True))

print(f"data_set.shape = {data_set.shape}")

features, labels, z = genFeaturesLabels(
    data_set, num_points=50, maxFr=0.99, maxHoverB=-0.7
)

a= len(features[0,0, :])
b   = len(features[0])
net = ResNetGELU(in_channels=b)

trainName = "ResNetdatapro3S_Minus07_noise_enhancement"
trainpro(trainName, net, num_points=50, maxh=10, maxFr=0.99, maxHoverB=-0.7)

In [ ]:
NAME='ResNetdatapro3S_Minus07_noise_enhancement'
lossName = NAME +'_loss.log'
data=np.loadtxt(lossName,unpack=True)
plt.semilogy(data[1])
plt.semilogy(data[2])